In [1]:
!pip install mistralai chromadb "psycopg[binary]" "fastapi[standard]" -q

from mistralai.client import Mistral
import chromadb
from chromadb import Documents, EmbeddingFunction, Embeddings
from chromadb.utils.embedding_functions import register_embedding_function
import psycopg
import os

In [2]:
MISTRAL_API_KEY = os.getenv("MISTRAL_API_KEY")
POSTGRES_DB_URL = os.getenv("POSTGRES_DB_URL")

In [4]:
@register_embedding_function
class MistralEmbeddingFunction(EmbeddingFunction):

    def __init__(self):
        self.model = Mistral(api_key=MISTRAL_API_KEY) 

    def __call__(self, input: Documents) -> Embeddings:
        res = self.model.embeddings.create(model="mistral-embed", inputs=input)
        return [x.embedding for x in res.data] 

    @staticmethod
    def name() -> str:
        return "Mistral EF (API)"

In [5]:
# Create Collection
chroma_client = chromadb.PersistentClient(path="./comic_embeddings_db")
collection = chroma_client.get_or_create_collection(
    name = "comic_embeddings",
    embedding_function = MistralEmbeddingFunction()
)

In [5]:
# chroma_client.delete_collection("comic_embeddings")

In [6]:
# Add embeddings for all comics
comic = []
with psycopg.connect(POSTGRES_DB_URL) as conn:
    with conn.cursor() as cur:
        cur.execute("select * from comic")
        for row in cur:
            comic.append(row)
        
comic_documents = []
comic_id = []
for c in comic:
    comic_documents.append(c[4])
    comic_id.append(str(c[0]))

collection.upsert(
    ids=comic_id,
    documents=comic_documents
)

In [6]:
collection.query(
    query_texts=["ghoul"],
    n_results = 5
)

{'ids': [['2258', '2103', '2302', '1953', '2160']],
 'embeddings': None,
 'documents': [['Lurking within the shadows of Tokyo are frightening beings known as "ghouls," who satisfy their hunger by feeding on humans once night falls. Ken Kaneki, an unsuspecting university freshman, finds himself caught in a world between humans and ghouls when his date turns out to be a ghoul after his flesh.',
   "After being aggressively rejected, Momo Ayase finds herself sulking when she stumbles across a boy being bullied. Saved by her rash kindness, the occult-obsessed boy attempts to speak to her about supernatural interests he believes they share. Rejecting his claims, Ayase proclaimed that she is instead a believer in ghosts, starting an argument between the two over which is real. In a bet to determine who's correct, the two decide to separately visit locations associated with both the occult and the supernatural—Ayase visiting the former and the boy visiting the latter. When the two reach their

In [7]:
embedding_1 = collection.get(ids = ['2258'], include=["embeddings"])["embeddings"][0]
print("len(embedding) =", len(embedding_1))

len(embedding) = 1024
